# 02 - Eksperimen Klasik (Skenario 1-9)

SVM & Random Forest dengan feature engineering manual. Tiap skenario
mengubah **satu variabel** untuk isolasi efek (restorasi, enhancement,
segmentasi, fitur, classifier). Jalankan **berurutan** setelah `01_eda.ipynb`.

In [1]:
# ============================================================
# Setup cell - Kaggle Notebooks (Kaggle-only). Jalankan PALING ATAS.
# Cara attach dataset: panel kanan > + Add Data > cari
#   'fruit and vegetable disease healthy vs rotten' > Add.
# ============================================================
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import sys
import shutil
import subprocess
from pathlib import Path

# 1. Clone repo dari GitHub (atau pull jika sudah ada di sesi ini)
REPO_URL = "https://github.com/faizhuda/pcd-kelompok-17.git"
PROJECT_DIR = Path("/kaggle/working/pcd-kelompok-17")
if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=False)

# 2. Working directory ke root project + tambah ke sys.path
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# 3. Dependency inti SUDAH pre-installed di Kaggle -> tidak ada pip install.

# 4. Dataset gambar (read-only, hasil + Add Data)
# Auto-detect: Kaggle bisa mount di /kaggle/input/<slug> atau
# /kaggle/input/datasets/<user>/<slug> tergantung cara attach.
_DATASET_SLUG = 'fruit-and-vegetable-disease-healthy-vs-rotten'
_candidates = [
    Path('/kaggle/input') / _DATASET_SLUG,
    Path('/kaggle/input/datasets/muhammad0subhan') / _DATASET_SLUG,
]
RAW_DATA_DIR = next((p for p in _candidates if p.exists()), None)
if RAW_DATA_DIR is None:
    # Fallback: cari folder mana saja di /kaggle/input yang berisi gambar dataset
    for _p in Path('/kaggle/input').rglob(_DATASET_SLUG):
        if _p.is_dir():
            RAW_DATA_DIR = _p
            break
assert RAW_DATA_DIR is not None, "Dataset belum di-attach. + Add Data > cari dataset > Add."

# 5. Auto-restore hasil notebook sebelumnya (untuk notebook 03 & 04).
#    Attach output run lama via: + Add Data > Your Work / Dataset bersama.
def restore_previous_outputs():
    # Kaggle mounts notebook outputs di /kaggle/input/notebooks/<user>/<notebook>/
    # sehingga perlu rglob, bukan glob satu level.
    restored = []
    for repo in Path("/kaggle/input").rglob("pcd-kelompok-17"):
        if not repo.is_dir():
            continue
        for sub in ("results", "data/processed"):
            src_dir = repo / sub
            if src_dir.exists():
                shutil.copytree(src_dir, PROJECT_DIR / sub, dirs_exist_ok=True)
                restored.append(str(src_dir))
    return restored

restored = restore_previous_outputs()
print("Project :", PROJECT_DIR)
print("Dataset :", RAW_DATA_DIR)
print("Restore :", restored or "(mulai dari nol)")


Cloning into '/kaggle/working/pcd-kelompok-17'...


Project : /kaggle/working/pcd-kelompok-17
Dataset : /kaggle/input/datasets/muhammad0subhan/fruit-and-vegetable-disease-healthy-vs-rotten
Restore : (mulai dari nol)


In [2]:
import os
import sys
from pathlib import Path

# Setup cell sudah chdir ke PROJECT_DIR & menambah sys.path (Kaggle-only).
ROOT = Path("/kaggle/working/pcd-kelompok-17")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.experiments import (
    run_classical_scenario,
    run_mcnemar_pair,
    select_best_enhancement,
)
from src.utils import build_dataset_index, get_project_paths, make_splits, set_seed

set_seed(42)
paths = get_project_paths()
# Split di-regenerate dari dataset (deterministik, SEED=42) - tidak baca splits.json
# RAW_DATA_DIR sudah di-set setup cell (auto-detect path Kaggle)
train, val, test = make_splits(build_dataset_index(RAW_DATA_DIR))
cache_dir = paths["data_processed"]
metrics_dir = paths["metrics"]
figures_dir = paths["figures_confusion"]
models_dir = paths["models"]
print(len(train), len(val), len(test))


E0000 00:00:1780693527.774240      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1780693527.861860      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1780693528.557477      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780693528.557540      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780693528.557546      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1780693528.557550      16 computation_placer.cc:177] computation placer already registered. Please check linka

20503 4394 4394


## Skenario 1-4: Baseline, Restorasi (SSR), Enhancement

- **S1** = baseline mentah (tanpa restorasi, tanpa enhancement)
- **S2** = + restorasi SSR (isolasi efek koreksi pencahayaan vs S1)
- **S3/S4** = SSR + CLAHE / gamma. E* dipilih dari S2-S4 (val F1).

In [3]:
val_f1_map = {}
scenario_results = {}

for sid in range(1, 5):
    print(f"\n=== Skenario {sid} ===")
    res = run_classical_scenario(
        sid, train, val, test,
        metrics_dir, figures_dir, models_dir, cache_dir,
    )
    scenario_results[sid] = res
    # E* dipilih di antara skenario ber-SSR (S2 none, S3 clahe, S4 gamma).
    # S1 = baseline mentah (tanpa restorasi) -> tidak ikut pemilihan enhancement.
    if sid >= 2:
        val_f1_map[res["enhancement"]] = res["val_f1"]
    print(f"Val F1: {res['val_f1']:.4f} | Test F1: {res['test_metrics']['f1_weighted']:.4f}")

best_enh = select_best_enhancement(val_f1_map, metrics_dir)
print(f"\nE* (enhancement terbaik): {best_enh}")



=== Skenario 1 ===


Extracting features: 100%|██████████| 4394/4394 [05:52<00:00, 12.47it/s]


Val F1: 0.9715 | Test F1: 0.9704

=== Skenario 2 ===


Extracting features: 100%|██████████| 4394/4394 [05:26<00:00, 13.45it/s]


Val F1: 0.9490 | Test F1: 0.9477

=== Skenario 3 ===


Extracting features: 100%|██████████| 4394/4394 [06:02<00:00, 12.11it/s]


Val F1: 0.9531 | Test F1: 0.9561

=== Skenario 4 ===


Extracting features: 100%|██████████| 4394/4394 [05:42<00:00, 12.83it/s]


Val F1: 0.9518 | Test F1: 0.9492

E* (enhancement terbaik): clahe


## Skenario 5-9: Segmentasi, Ablasi Fitur, Random Forest

- **S5** = E* + segmentasi, semua fitur, SVM (pipeline klasik penuh)
- **S6/S7/S8** = ablasi fitur (warna saja / tekstur saja / bentuk saja)
- **S9** = S5 dengan Random Forest (perbandingan classifier + feature importance)

In [4]:
for sid in range(5, 10):
    print(f"\n=== Skenario {sid} ===")
    res = run_classical_scenario(
        sid, train, val, test,
        metrics_dir, figures_dir, models_dir, cache_dir,
    )
    scenario_results[sid] = res
    print(f"Test F1: {res['test_metrics']['f1_weighted']:.4f}")



=== Skenario 5 ===


Extracting features: 100%|██████████| 4394/4394 [07:12<00:00, 10.16it/s]


Test F1: 0.9529

=== Skenario 6 ===


Extracting features: 100%|██████████| 4394/4394 [02:10<00:00, 33.76it/s]


Test F1: 0.9356

=== Skenario 7 ===


Extracting features: 100%|██████████| 4394/4394 [05:18<00:00, 13.78it/s]


Test F1: 0.8101

=== Skenario 8 ===


Extracting features: 100%|██████████| 4394/4394 [01:54<00:00, 38.27it/s]


Test F1: 0.5958

=== Skenario 9 ===
Test F1: 0.9379


## Uji Signifikansi McNemar (isolasi tiap tahap)

In [5]:
from src.utils import read_best_enhancement

best_enh = read_best_enhancement(metrics_dir)
# Skenario no-seg yang memakai E* (untuk isolasi efek segmentasi):
# none -> S2, clahe -> S3, gamma -> S4.
enh_noseg_sid = {"none": 2, "clahe": 3, "gamma": 4}[best_enh]
y_true = scenario_results[1]["y_test"]

# 1. Efek restorasi SSR: S2 (ssr) vs S1 (mentah)
run_mcnemar_pair(
    "S2 vs S1 (SSR)", "S2", "S1",
    y_true, scenario_results[2]["y_pred"], scenario_results[1]["y_pred"], metrics_dir,
)

# 2. Efek enhancement E*: S{E*} vs S2 (hanya bermakna bila E* != none)
if enh_noseg_sid != 2:
    run_mcnemar_pair(
        f"E*({best_enh}) vs S2", f"S{enh_noseg_sid}", "S2",
        y_true, scenario_results[enh_noseg_sid]["y_pred"], scenario_results[2]["y_pred"], metrics_dir,
    )

# 3. Efek segmentasi: S5 (E*+seg) vs E* tanpa seg
run_mcnemar_pair(
    "S5 vs E*-noseg (segmentasi)", "S5", f"S{enh_noseg_sid}",
    y_true, scenario_results[5]["y_pred"], scenario_results[enh_noseg_sid]["y_pred"], metrics_dir,
)
print("McNemar CNN (S10/S11 vs S5) dijalankan di notebook 03.")


McNemar CNN (S10/S11 vs S5) dijalankan di notebook 03.


In [6]:
import pandas as pd
sig_path = metrics_dir / "significance_tests.csv"
if sig_path.exists():
    display(pd.read_csv(sig_path))


,comparison,model_a,model_b,statistic,p_value,conclusion
0,S2 vs S1 (SSR),S2,S1,44.550000,2.479435e-11,signifikan
1,E*(clahe) vs S2,S3,S2,8.697987,3.185617e-03,signifikan
2,S5 vs E*-noseg (segmentasi),S5,S3,1.564815,2.109616e-01,tidak signifikan
